In [ ]:
# Portability: target catalog and source data location are parameters.
# Defaults reproduce the original setup (Databricks Free Edition); override them via widgets or job parameters.
import re
dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("source_schema", "samples.tpch", "Source data (catalog.schema)")
CATALOG = dbutils.widgets.get("catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", SOURCE_SCHEMA):
    raise ValueError(f"invalid source_schema: {SOURCE_SCHEMA!r}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  source_schema={SOURCE_SCHEMA}")

In [0]:
# TPC-H README for exploration only; skipped when the workspace has no /dbfs sample datasets
try:
    with open("/dbfs/databricks-datasets/tpch/README.md", "r") as f:
        content = f.read()
        displayHTML(f"<pre>{content}</pre>")
except OSError as e:
    print(f"TPC-H README not available in this workspace ({e}); continuing.")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bda_bronze")

In [0]:
from pyspark.sql.functions import current_timestamp

tpch_tables = [
    "customer", "lineitem", "nation", "orders", 
    "part", "partsupp", "region", "supplier"
]

def load_tables(tables: list[str], database: str):
    """
    Load tables from the tpch database to the bronze layer
    Args:
        tables (list[str]): list of tables to load
        database (str): database name
    """
    for table in tables:
        print(f" | Current table: {table}")
        df = spark.read.table(f"{SOURCE_SCHEMA}.{table}")

        df_bronze = df.withColumn("ingestion_timestamp", current_timestamp())

        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{database}.{table}")

load_tables(tpch_tables, "bda_bronze")


In [0]:
%sql
SELECT * FROM bda_bronze.orders LIMIT 5;